## Severity Analysis Overview

This script performs mental health analysis using machine learning. It starts by loading a dataset of Reddit posts (`mergeddata.csv`) and cleaning the `Selftext` column by lowercasing, removing punctuation, tokenizing, and removing stopwords. It uses keyword matching to simulate disorder labels (Anxiety, Depression, ADHD) and assigns a severity score based on the presence of intensifiers, qualifiers, and medication mentions.

The processed text is then vectorized using TF-IDF to extract features, and a Random Forest Classifier is trained to predict mental health disorders, while a Random Forest Regressor estimates severity levels. The model's performance is evaluated using classification metrics and regression accuracy. Finally, the script applies the models to sample inputs, generates predictions for the entire dataset, and saves the enhanced results (with predicted disorder and severity) to `mergeddata_with_ml_predictions.csv`.

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('/content/mergeddata.csv')

In [ ]:
df.head()

## Severity Model

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import classification_report, mean_squared_error, r2_score
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re


nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')


df = pd.read_csv('mergeddata.csv')
df['Selftext'] = df['Selftext'].fillna('')


def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    return ' '.join(tokens)

df['Processed_Selftext'] = df['Selftext'].apply(preprocess_text)

anxiety_keywords = {
    "chest pain", "chest tightness", "lightheadedness", "nausea", "often", "racing thoughts",
    "can't stop", "scared", "insomnia", "anxiety attack", "headaches", "can't stop worrying",
    "shaking", "can't sleep", "bathroom issues", "diarrhea", "dizzy", "health anxiety",
    "vomit", "throwing up", "panic attack", "everyday", "extreme", "terrified", "heart racing",
    "benzos", "nervous breakdown", "agoraphobia"
}

depression_keywords = {
    "sad", "down", "lonely", "die", "dying", "kill myself", "hopeless", "worthless",
    "alone", "pain", "can’t take this", "giving up", "exhausted", "crying", "numb",
    "losing weight", "gaining weight"
}

adhd_keywords = {
    "impulsive", "impulsivity", "keep track of", "losing things", "medication",
    "stimulants", "motivation", "procrastination", "productivity", "focus", "planning",
    "task paralysis", "forgetful", "restlessness", "distracted", "disorganized",
    "fidgeting", "interrupting"
}

def simulate_training_disorder(selftext, subreddit):
    text_lower = str(selftext).lower()
    depression_score = sum(1 for word in depression_keywords if word in text_lower)
    adhd_score = sum(1 for word in adhd_keywords if word in text_lower)
    anxiety_score = sum(1 for word in anxiety_keywords if word in text_lower)

    if depression_score > 0:
        return "Depression"
    elif adhd_score > 0 and subreddit == "ADHD":
        return "ADHD"
    elif anxiety_score > 0 and subreddit == "Anxiety":
        return "Anxiety"
    else:
        return subreddit if subreddit in ["ADHD", "Anxiety"] else "Depression"


def simulate_severity(selftext, disorder):
    intensifiers = ["severe", "constant", "every day", "terrified", "cant stop", "extreme", "everyday"]
    qualifiers = ["mild", "sometimes", "occasional"]
    medications = ["xanax", "prozac", "zoloft", "vyvanse", "adderall", "ssri", "lexapro",
                   "concerta", "ritalin", "propranolol", "benzos", "stimulants", "medication"]

    text_lower = str(selftext).lower()
    if disorder == "Anxiety":
        min_score, max_score = 5, 10
        keywords = anxiety_keywords
    elif disorder == "Depression":
        min_score, max_score = 4, 10
        keywords = depression_keywords
    elif disorder == "ADHD":
        min_score, max_score = 4, 10
        keywords = adhd_keywords
    else:
        raise ValueError("Disorder must be Anxiety, Depression, or ADHD")

    score = 1
    keyword_count = sum(1 for word in keywords if word in text_lower)
    if keyword_count > 0:
        score = min_score
        if keyword_count >= 3:
            score = min_score + 2

    for intensifier in intensifiers:
        if intensifier in text_lower:
            score = min(score + 1, max_score)
            break
    for qualifier in qualifiers:
        if qualifier in text_lower:
            score = max(score - 1, min_score)
            break

    if any(med in text_lower for med in medications):
        score = max(score, min_score + 1)

    return max(min_score, min(score, max_score))


df['Training_Disorder'] = df.apply(
    lambda row: simulate_training_disorder(row['Selftext'], row['Subreddit']), axis=1
)
df['Simulated_Severity'] = df.apply(
    lambda row: simulate_severity(row['Selftext'], row['Training_Disorder']), axis=1
)


vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 3))
X_disorder = vectorizer.fit_transform(df['Processed_Selftext'])
y_disorder = df['Training_Disorder']


X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(X_disorder, y_disorder, test_size=0.2, random_state=42)


clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_d, y_train_d)


y_pred_d = clf.predict(X_test_d)
print("Disorder Prediction Evaluation:")
print(classification_report(y_test_d, y_pred_d))
disorder_accuracy = np.mean(y_pred_d == y_test_d)
print(f"Disorder Prediction Accuracy: {disorder_accuracy:.2%}")


X_severity = X_disorder
y_severity = df['Simulated_Severity']


X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_severity, y_severity, test_size=0.2, random_state=42)


regressor = RandomForestRegressor(n_estimators=100, random_state=42)
regressor.fit(X_train_s, y_train_s)


y_pred_s = regressor.predict(X_test_s)
mse = mean_squared_error(y_test_s, y_pred_s)
r2 = r2_score(y_test_s, y_pred_s)
severity_accuracy = np.mean(np.abs(y_test_s - y_pred_s) <= 1)
print(f"Severity Prediction Mean Squared Error: {mse:.2f}")
print(f"Severity Prediction R-squared: {r2:.2f}")
print(f"Severity Prediction Pseudo-Accuracy (within ±1): {severity_accuracy:.2%}")


def predict_disorder_and_severity(selftext):
    processed_text = preprocess_text(selftext)
    X_new = vectorizer.transform([processed_text])


    disorder = clf.predict(X_new)[0]
    severity = regressor.predict(X_new)[0]

    if disorder == "Anxiety":
        severity = max(5, min(10, severity))
    elif disorder in ["Depression", "ADHD"]:
        severity = max(4, min(10, severity))

    return disorder, round(severity)


test_indices = [0, 1, 2, 10, 20]
for idx in test_indices:
    selftext = df['Selftext'].iloc[idx]
    subreddit = df['Subreddit'].iloc[idx]
    disorder, severity = predict_disorder_and_severity(selftext)
    print(f"Selftext: {selftext[:50]}...")
    print(f"Actual Subreddit: {subreddit}")
    print(f"Predicted Disorder: {disorder}")
    print(f"Predicted Severity: {severity}\n")

df[['Predicted_Disorder', 'Predicted_Severity']] = df['Selftext'].apply(
    lambda x: pd.Series(predict_disorder_and_severity(x))
)
df.to_csv('mergeddata_with_ml_predictions.csv', index=False)
print("Predictions saved to 'mergeddata_with_ml_predictions.csv'")

In [ ]:
df_new = pd.read_csv('/content/mergeddata_with_ml_predictions.csv')

In [ ]:
pd.reset_option('display.max_colwidth')

In [ ]:
df_new[['Selftext', 'Subreddit','Training_Disorder', 'Predicted_Disorder'
,'Simulated_Severity', 'Predicted_Severity']].head(5)


In [ ]:
df_new = df_new.drop(columns=['Training_Disorder', 'Simulated_Severity'])

In [ ]:
df_new = pd.read_csv('/content/mergeddata_with_ml_predictions.csv')
df_new[['Selftext', 'Subreddit', 'Predicted_Disorder', 'Predicted_Severity']].head(5)

In [ ]:
df_new[df_new['Subreddit'] == 'depression'][['Selftext', 'Subreddit', 'Predicted_Disorder', 'Predicted_Severity']].head(5)

In [ ]:
df_new[df_new['Subreddit'] == 'ADHD'][['Selftext', 'Subreddit', 'Predicted_Disorder', 'Predicted_Severity']].head(5)

In [ ]:
df_new[['Selftext', 'Subreddit', 'Predicted_Disorder', 'Predicted_Severity']].head(5)

In [ ]:
df_new[['Selftext', 'Subreddit', 'Predicted_Disorder', 'Predicted_Severity']].head(5)

In [ ]:
df_new[df_new['Subreddit'] == 'ADHD'].head(10)

In [ ]:
df_new[df_new['Subreddit'] == 'depression'].head(10)

In [ ]:
df_new[(df_new["Predicted_Severity"] > 5) & (df["Subreddit"] == "depression")].head()